# Domain decomposition

How mpiarray splits an array over the ranks of an MPI job, step by step: the process grid,
the cut points along each axis, the numbering of the ranks and their neighbours, and the
halo cells around every block. The [Layouts](/mpiarray/guides/layouts/) guide is the
reference for the `Layout` API; this page explains the decomposition itself, with figures
drawn from mpiarray's own layouts, and ends with an interactive explorer.

The figures describe runs on 6, 12 or 16 ranks without starting them: a `Layout` only asks
its communicator for the number of ranks and its own rank, so a small stand-in communicator
is enough to compute what every rank of such a run would own (it cannot communicate). Every
number in the figures comes from `Layout`.

In [ ]:
import matplotlib.pyplot as plt
import maybempi
import numpy as np
from matplotlib.colors import to_rgba
from matplotlib.patches import Rectangle

import mpiarray as mpa
from mpiarray import Layout

comm = mpa.default_comm()
rank, nranks = comm.Get_rank(), comm.Get_size()
PROC_NULL = maybempi.get_mpi().PROC_NULL  # the "neighbour" beyond a wall


class Ranks:
    """A stand-in communicator: the two things `Layout` reads to compute a decomposition.

    Layouts built on it describe what a run on ``size`` ranks would own, without starting
    one. They cannot communicate.
    """

    def __init__(self, size, rank=0):
        self.size, self.rank = size, rank

    def Get_size(self):
        return self.size

    def Get_rank(self):
        return self.rank


def layouts(size, shape, **options):
    """One `Layout` per rank of a run on ``size`` ranks, each knowing its own neighbours."""
    return [Layout(shape, comm=Ranks(size, r), **options) for r in range(size)]


# one colour per rank, the same as in the explorer at the end of the page
PALETTE = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
    "#aec7e8",
    "#ffbb78",
    "#98df8a",
    "#ff9896",
    "#c5b0d5",
    "#c49c94",
    "#f7b6d2",
    "#c7c7c7",
    "#dbdb8d",
    "#9edae5",
]
ACCENT = "#2a78d6"
WALL = {
    "facecolor": "#ededed",
    "hatch": "///",
    "edgecolor": "#9a9a9a",
    "linewidth": 0.5,
}


def rank_color(r):
    return PALETTE[r % len(PALETTE)]


def owner_map(layout):
    """The rank that owns every cell of a 2-D layout, as an array of the global shape."""
    n0, n1 = layout.shape
    i, j = np.meshgrid(np.arange(n0), np.arange(n1), indexing="ij")
    return layout.owners(np.stack([i, j], axis=-1))


def caption(ax, text):
    ax.text(0.5, -0.04, text, transform=ax.transAxes, ha="center", va="top", fontsize=9)


def draw_blocks(
    ax, layout, *, coords=False, cells=True, caption_text=None, fontsize=11
):
    """Colour the cells of a 2-D array by the rank that owns them and draw the cuts."""
    n0, n1 = layout.shape
    if layout.replicated:
        image = np.broadcast_to(to_rgba("#d8d8d8", 0.8), (n0, n1, 4))
    else:
        colors = np.array([to_rgba(rank_color(r), 0.65) for r in range(layout.size)])
        image = colors[owner_map(layout)]
    ax.imshow(image, extent=(0, n1, n0, 0), interpolation="nearest")
    if cells:
        ax.set_xticks(np.arange(n1 + 1), minor=True)
        ax.set_yticks(np.arange(n0 + 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.6)
    ax.grid(which="major", visible=False)
    ax.hlines(layout.bounds[0], 0, n1, color="black", linewidth=1.8)
    ax.vlines(layout.bounds[1], 0, n0, color="black", linewidth=1.8)
    ax.set_xticks(
        layout.bounds[1]
    )  # where the blocks start along each axis, and the length
    ax.set_yticks(layout.bounds[0])
    ax.tick_params(which="both", length=0)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    if layout.replicated:
        ax.text(
            n1 / 2, n0 / 2, "every rank", ha="center", va="center", fontsize=fontsize
        )
    else:
        for r in range(layout.size):
            (a0, a1), (b0, b1) = layout.index_bounds_of(r)
            label = str(r) + (f"\n{layout.coord_of(r)}" if coords else "")
            x, y = (b0 + b1) / 2, (a0 + a1) / 2
            ax.text(x, y, label, ha="center", va="center", fontsize=fontsize)
    ax.set(xlabel="axis 1", ylabel="axis 0", xlim=(0, n1), ylim=(n0, 0))
    if caption_text:
        caption(ax, caption_text)


def draw_exploded(ax, layout, gap=1.5, fontsize=9):
    """Every rank's block on its own, placed on the process grid with gaps between them."""
    n0, n1 = layout.shape
    for r in range(layout.size):
        (a0, a1), (b0, b1) = layout.index_bounds_of(r)
        c0, c1 = layout.coord_of(r)
        x, y, w, h = b0 + c1 * gap, a0 + c0 * gap, b1 - b0, a1 - a0
        ax.add_patch(Rectangle((x, y), w, h, facecolor=rank_color(r), alpha=0.65))
        for j in range(w + 1):
            ax.plot([x + j, x + j], [y, y + h], color="white", linewidth=0.6)
        for i in range(h + 1):
            ax.plot([x, x + w], [y + i, y + i], color="white", linewidth=0.6)
        ax.add_patch(
            Rectangle((x, y), w, h, fill=False, edgecolor="black", linewidth=1.5)
        )
        label = f"rank {r}\n{layout.local_shape_of(r)}"
        ax.text(
            x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=fontsize
        )
    g0, g1 = layout.process_grid
    ax.set(
        xlim=(-0.5, n1 + (g1 - 1) * gap + 0.5), ylim=(n0 + (g0 - 1) * gap + 0.5, -0.5)
    )
    ax.set_aspect("equal")
    ax.axis("off")


def halo_boxes(layout):
    """The boxes of halo cells of this rank, in global indices, and whether each is beyond a wall.

    Along a periodic axis a box is wrapped to the other side of the array.
    """
    boxes = []
    for d0 in (-1, 0, 1):
        for d1 in (-1, 0, 1):
            if d0 == d1 == 0:
                continue
            box, wall = [], False
            axes = zip(
                (d0, d1),
                layout.index_bounds,
                layout.halo,
                layout.shape,
                layout.periodic,
                strict=True,
            )
            for d, (start, end), h, length, periodic in axes:
                lo, hi = (
                    (start, end)
                    if d == 0
                    else (start - h, start)
                    if d < 0
                    else (end, end + h)
                )
                if lo == hi:
                    break
                if lo < 0 or hi > length:
                    if periodic:
                        lo, hi = (
                            (lo + length, hi + length)
                            if lo < 0
                            else (lo - length, hi - length)
                        )
                    else:
                        wall = True
                box.append((lo, hi))
            else:
                boxes.append((tuple(box), wall))
    return boxes


def draw_halo(ax, layout):
    """Mark this rank's halo cells in the global array: dashed where a neighbour fills them, hatched beyond a wall."""
    for ((r0, r1), (c0, c1)), wall in halo_boxes(layout):
        if wall:
            style = WALL
        else:
            style = {
                "facecolor": ACCENT,
                "alpha": 0.3,
                "edgecolor": ACCENT,
                "linestyle": "--",
            }
        ax.add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, **style))
    (a0, a1), (b0, b1) = layout.index_bounds
    ax.add_patch(
        Rectangle((b0, a0), b1 - b0, a1 - a0, fill=False, edgecolor=ACCENT, linewidth=3)
    )
    (n0, n1), (h0, h1) = layout.shape, layout.halo
    ax.set(xlim=(-h1 - 0.5, n1 + h1 + 0.5), ylim=(n0 + h0 + 0.5, -h0 - 0.5))


def storage_sources(layout):
    """The rank each cell of this rank's storage comes from: itself, a neighbour, or -1 beyond a wall."""
    starts = [
        start - h
        for (start, _), h in zip(layout.index_bounds, layout.halo, strict=True)
    ]
    axes = [
        np.arange(s, s + n) for s, n in zip(starts, layout.storage_shape, strict=True)
    ]
    index = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1)
    inside = np.ones(layout.storage_shape, dtype=bool)
    for axis, (length, periodic) in enumerate(
        zip(layout.shape, layout.periodic, strict=True)
    ):
        if periodic:
            index[..., axis] %= length
        else:
            inside &= (index[..., axis] >= 0) & (index[..., axis] < length)
    sources = np.full(layout.storage_shape, -1)
    sources[inside] = layout.owners(index[inside])
    return sources


def draw_storage(ax, layout, fontsize=10):
    """This rank's storage: its block framed by the halo cells, coloured by the rank they come from."""
    sources = storage_sources(layout)
    (m0, m1), (h0, h1) = layout.storage_shape, layout.halo
    image = np.zeros((m0, m1, 4))
    for r in np.unique(sources):
        image[sources == r] = (
            to_rgba(rank_color(r), 0.65) if r >= 0 else to_rgba("white")
        )
    ax.imshow(image, extent=(0, m1, m0, 0), interpolation="nearest")
    for i, j in zip(*np.nonzero(sources < 0), strict=True):
        ax.add_patch(Rectangle((j, i), 1, 1, **WALL))
    ax.set_xticks(np.arange(m1 + 1), minor=True)
    ax.set_yticks(np.arange(m0 + 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.6)
    ax.grid(which="major", visible=False)
    block = Rectangle(
        (h1, h0), m1 - 2 * h1, m0 - 2 * h0, fill=False, edgecolor="black", linewidth=2
    )
    ax.add_patch(block)
    ax.add_patch(
        Rectangle((0, 0), m1, m0, fill=False, edgecolor="black", linestyle="--")
    )
    ax.set_xticks(sorted({0, h1, m1 - h1, m1}))
    ax.set_yticks(sorted({0, h0, m0 - h0, m0}))
    ax.tick_params(which="both", length=0)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set(xlabel="storage index, axis 1", ylabel="storage index, axis 0")
    ax.text(
        m1 / 2,
        m0 / 2,
        f"local\nrank {layout.rank}",
        ha="center",
        va="center",
        fontsize=fontsize,
    )


def draw_neighbours(ax, layout, margin=2.5):
    """Arrows from this rank's block to its neighbours along each axis, leaving the array where they wrap around."""
    (a0, a1), (b0, b1) = layout.index_bounds
    n0, n1 = layout.shape
    ax.add_patch(
        Rectangle((b0, a0), b1 - b0, a1 - a0, fill=False, edgecolor=ACCENT, linewidth=3)
    )
    arrow = {
        "arrowstyle": "-|>",
        "color": "black",
        "linewidth": 1.5,
        "shrinkA": 0,
        "shrinkB": 0,
    }
    out0, out1 = (
        margin - 0.7,
        margin - 0.7,
    )  # how far the wrap-around arrows leave the array
    tags = {}
    for axis, pair in enumerate(layout.neighbours):
        for step, neighbour in zip((-1, 1), pair, strict=True):
            if neighbour == PROC_NULL:
                x, y = (b0 + b1) / 2, (a0 + a1) / 2
                if axis == 0:
                    y = -out0 if step < 0 else n0 + out0
                else:
                    x = -out1 if step < 0 else n1 + out1
                rotation = 0 if axis == 0 else 90
                ax.text(
                    x,
                    y,
                    "PROC_NULL",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="gray",
                    rotation=rotation,
                )
                continue
            (c0, c1), (d0, d1) = layout.index_bounds_of(neighbour)
            if axis == 0:
                x = (b0 + b1) / 2
                start, end = (
                    (x, a0 if step < 0 else a1),
                    ((d0 + d1) / 2, c1 if step < 0 else c0),
                )
                adjacent = (c1 == a0) if step < 0 else (c0 == a1)
                exit_point = (x, -out0 if step < 0 else n0 + out0)
                entry = (end[0], n0 + out0 if step < 0 else -out0)
            else:
                y = (a0 + a1) / 2
                start, end = (
                    (b0 if step < 0 else b1, y),
                    (d1 if step < 0 else d0, (c0 + c1) / 2),
                )
                adjacent = (d1 == b0) if step < 0 else (d0 == b1)
                exit_point = (-out1 if step < 0 else n1 + out1, y)
                entry = (n1 + out1 if step < 0 else -out1, end[1])
            if adjacent:
                ax.annotate("", xy=end, xytext=start, arrowprops=arrow)
            else:  # wrap-around: out of the array on this side, in again on the other
                ax.annotate("", xy=exit_point, xytext=start, arrowprops=arrow)
                ax.annotate("", xy=end, xytext=entry, arrowprops=arrow)
            if neighbour != layout.rank:
                tags.setdefault(neighbour, []).append(
                    f"axis {axis} {'−' if step < 0 else '+'}"
                )
    for neighbour, names in tags.items():
        (c0, c1), (d0, d1) = layout.index_bounds_of(neighbour)
        ax.text(
            (d0 + d1) / 2,
            (c0 + c1) / 2 + 2.2,
            ", ".join(names),
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
        )
    ax.set(xlim=(-margin, n1 + margin), ylim=(n0 + margin, -margin))

## One array, one block per rank

A distributed array is one global array of which every rank stores one block: a box of
global indices, `layout.index_bounds`, with nothing in common with the other ranks' boxes.
The blocks tile the array, so every cell has exactly one owner, `layout.owner(index)`. Here
a $12 \times 9$ array is split over 6 ranks along both axes:

In [ ]:
layout = layouts(6, (12, 9), split=(0, 1))[0]  # what rank 0 of a run on 6 ranks sees

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4.4))
draw_blocks(left, layout, caption_text="the global array, coloured by owner")
draw_exploded(right, layout)
caption(right, "the block each rank stores (its local_shape)")
plt.show()

The decomposition is decided in two steps, each explained below:

1. **The process grid**: how many ranks go along each axis ($2 \times 3$ here), so that the
   ranks form a grid of the same number of dimensions as the array.
2. **The cut points**: along every axis, where one rank's index range ends and the next
   begins (`layout.bounds`, here `(0, 6, 12)` and `(0, 3, 6, 9)`). A rank's block is the
   product of one piece per axis.

Then every rank is given a position on the grid, from which its neighbours follow, and
stores its block with a frame of halo cells.

## The process grid

`split` names the axes to cut; the default is axis 0 only. The ranks are shared among the
split axes as evenly as divisibility allows, by `mpa.process_grid`, which only looks at the
number of ranks and the split axes, not at the shape. On two split axes:

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(9, 9.6))
for size, ax in zip(range(1, 17), axes.flat, strict=True):
    grid = layouts(size, (24, 24), split=(0, 1))[0]
    draw_blocks(ax, grid, cells=False, fontsize=8)
    g0, g1 = grid.process_grid
    ax.set(xlabel=None, ylabel=None, xticks=[], yticks=[])
    caption(ax, f"{size} ranks: {g0} × {g1}")
fig.tight_layout()
plt.show()

The split axes are filled in order. With $d$ split axes left to fill and $r$ ranks left to
place, the next axis gets the largest divisor of $r$ that is not above $\sqrt[d]{r}$, and the
last axis takes what remains, so the product is always the number of ranks. $12$ ranks give
$3 \times 4$, $16$ give $4 \times 4$; a prime number of ranks cannot be shared, so $7$ or
$13$ ranks all go to the last split axis. Rank counts with small factors ($4, 6, 8, 12, 16,
\dots$) give balanced grids.

Which axes are split decides the shape of the blocks. On 6 ranks, the same $12 \times 9$
array with `split=0`, `split=1`, `split=(0, 1)`, and `split=None`, where no axis is split
and every rank holds the whole array (a *replicated* layout):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 3.8))
for ax, split in zip(axes, (0, 1, (0, 1), None), strict=True):
    grid = layouts(6, (12, 9), split=split)[0]
    draw_blocks(ax, grid, fontsize=9)
    caption(ax, f"split={split}: process grid {grid.process_grid}")
    if ax is not axes[0]:
        ax.set(ylabel=None, yticks=[])
plt.show()

### Choosing the grid yourself

Because the automatic grid ignores the shape, an elongated array can get blocks that are
long and thin, with large faces between them. Every face is a surface through which halo
cells are exchanged at each `update_halos()`: the figure counts, for a $48 \times 24$ array
on 12 ranks with one halo layer, the cells every grid receives per update, summed over the
ranks. The automatic grid is $4 \times 3$; `process_grid=` chooses any other factorisation
(its product must be the number of ranks, and its axes with more than one rank become the
split axes):

In [ ]:
def halo_cells_received(size, shape, **options):
    """Cells that update_halos() fills on all ranks together: halo cells with a neighbour."""
    return sum(
        int(np.count_nonzero((sources != lay.rank) & (sources >= 0)))
        for lay in layouts(size, shape, **options)
        for sources in [storage_sources(lay)]
    )


fig, axes = plt.subplots(1, 6, figsize=(12, 4.2))
for ax, grid in zip(
    axes, [(12, 1), (6, 2), (4, 3), (3, 4), (2, 6), (1, 12)], strict=True
):
    lay = layouts(12, (48, 24), process_grid=grid, halo=1)[0]
    draw_blocks(ax, lay, cells=False, fontsize=8)
    received = halo_cells_received(12, (48, 24), process_grid=grid, halo=1)
    automatic = " (automatic)" if grid == (4, 3) else ""
    caption(ax, f"{grid[0]} × {grid[1]}{automatic}\n{received} halo cells")
    ax.set(xlabel=None, ylabel=None)
fig.tight_layout()
plt.show()

Blocks close to square exchange the least: $4 \times 3$ and $6 \times 2$ both give blocks of
$12 \times 8$ or $8 \times 12$ cells, while the strips of $12 \times 1$ and $1 \times 12$
exchange up to three times as much. Give the most ranks to the longest axis, and leave
unsplit the axes that are short or that an algorithm needs whole on every rank (a component
axis of a vector field, an axis you transform with an FFT). `halo=(2, 2, 0)` likewise gives
no halo cells along an axis that is not split.

## Cutting each axis

Along a split axis of length $L$ shared by $n$ ranks, rank $i$ (counted along that axis) gets
the half-open index range `mpa.chunk_bounds(L, n, i)`: every rank gets $\lfloor L / n
\rfloor$ elements, and the first $L \bmod n$ ranks one more, so the lengths differ by at most
one. An axis of 10 elements on 1 to 6 ranks:

In [ ]:
length = 10
fig, axes = plt.subplots(6, 1, figsize=(8, 4.4), sharex=True)
for n, ax in zip(range(1, 7), axes, strict=True):
    for i in range(n):
        start, end = mpa.chunk_bounds(length, n, i)
        ax.add_patch(
            Rectangle((start, 0), end - start, 1, facecolor=rank_color(i), alpha=0.65)
        )
        ax.add_patch(
            Rectangle((start, 0), end - start, 1, fill=False, edgecolor="black")
        )
        ax.text((start + end) / 2, 0.5, str(end - start), ha="center", va="center")
    ax.set(xlim=(0, length), ylim=(0, 1), yticks=[])
    ax.set_ylabel(f"{n} ranks", rotation=0, ha="right", va="center")
    ax.grid(visible=False)
axes[-1].set(xticks=range(length + 1), xlabel="global index")
plt.show()

The numbers in the blocks are their lengths: with 4 ranks, $10 = 3 + 3 + 2 + 2$. Each axis
is cut on its own, and `layout.bounds` collects the cut points of all axes: along each one,
the global index where every block starts, and the length. A block is the product of one
piece per axis, so the cuts of a $10 \times 7$ array on $2 \times 2$ ranks are two short
tuples:

In [ ]:
lay = layouts(4, (10, 7), split=(0, 1))[0]
print("process grid:", lay.process_grid)
print("bounds:      ", lay.bounds)
for r in range(4):
    print(
        f"rank {r} owns rows {lay.index_bounds_of(r)[0]} and columns {lay.index_bounds_of(r)[1]}"
    )

fig, ax = plt.subplots(figsize=(3.6, 4.4))
draw_blocks(ax, lay)
caption(ax, "ticks: where the blocks start, and the length")
plt.show()

### When the ranks do not fit

A split axis needs at least as many elements as ranks along it, and every halo must fit in
the smallest block of its axis (a halo is filled from the neighbouring block only).
Otherwise `Layout`, and every creation function, raises a `ValueError` on every rank rather
than leaving a rank without cells:

In [ ]:
try:
    Layout(
        (3, 40), comm=Ranks(16), split=0
    )  # only axis 0 may be cut: 3 elements, 16 ranks
except ValueError as error:
    print("ValueError:", error)
try:
    Layout(10, comm=Ranks(4), halo=3)  # blocks of 3, 3, 2 and 2 elements
except ValueError as error:
    print("ValueError:", error)

When the automatic grid is the problem, `Layout` first looks for another grid: if
`process_grid`'s choice gives an axis more ranks than elements, the layout takes, of all the
grids that fit, the one with the smallest halo surface. A $3 \times 40$ array on 16 ranks
with `split=(0, 1)` would get $4 \times 4$, which does not fit; of the grids that do,
$1 \times 16$ has the smallest faces and is chosen over $2 \times 8$:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 3.6))
for ax, grid in zip(axes, [None, (2, 8)], strict=True):
    options = {"split": (0, 1)} if grid is None else {"process_grid": grid}
    lay = layouts(16, (3, 40), halo=1, **options)[0]
    draw_blocks(ax, lay, fontsize=8)
    received = halo_cells_received(16, (3, 40), halo=1, **options)
    chosen = (
        "chosen: the smallest halo surface" if grid is None else "process_grid=(2, 8)"
    )
    caption(
        ax,
        f"{lay.process_grid[0]} × {lay.process_grid[1]}, {received} halo cells ({chosen})",
    )
    ax.set(xlabel=None, ylabel=None)
fig.tight_layout()
plt.show()

## Ranks on the grid

Every rank has a position on the process grid, `layout.process_coord`, and the ranks are
placed in row-major order: the last axis varies fastest, like the elements of a C-ordered
array. On a $3 \times 4$ grid, rank $r$ sits at coordinates $(r \,\text{div}\, 4,\; r \bmod
4)$, and the rank at $(c_0, c_1)$ is $4 c_0 + c_1$; `layout.coord_of` and `layout.rank_of`
convert between the two.

In [ ]:
lay = layouts(12, (24, 32), split=(0, 1))[0]
print("process grid:", lay.process_grid)
print("rank 5 sits at", lay.coord_of(5), "and", lay.rank_of((1, 1)), "sits at (1, 1)")
print("cell (7, 4) belongs to rank", lay.owner((7, 4)))

fig, ax = plt.subplots(figsize=(6, 5))
draw_blocks(ax, lay, coords=True, cells=False)
caption(ax, "rank and process coordinates of every block")
plt.show()

The rank is the one in the layout's communicator, `MPI.COMM_WORLD` by default, so the MPI
launcher decides which process gets which block. With `Layout(..., reorder=True)` the layout
asks MPI for a Cartesian communicator instead (`MPI_Cart_create` with reordering), which may
renumber the ranks so that neighbouring blocks land on nearby cores; the layout's `comm`,
`rank` and `neighbours` then refer to that communicator.

## Neighbours, periodicity and walls

`layout.neighbours` gives, for every axis, the pair of ranks whose blocks come just before
and just after this rank's along that axis. Inside the grid they are the adjacent blocks;
at the ends of an axis, `periodic` decides: along a periodic axis the neighbour is the
block at the other end of the array, and along a non-periodic one there is a *wall*, marked
by `MPI.PROC_NULL`. MPI calls with `PROC_NULL` do nothing, so halo code needs no special
case for walls.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6))
cases = [(5, (True, True)), (0, (True, True)), (0, (False, False))]
for ax, (r, periodic) in zip(axes, cases, strict=True):
    lay = layouts(12, (24, 32), split=(0, 1), periodic=periodic)[r]
    draw_blocks(ax, lay, cells=False, fontsize=9)
    draw_neighbours(ax, lay)
    caption(ax, f"rank {r}, periodic={periodic}\nneighbours {lay.neighbours}")
fig.tight_layout()
plt.show()

Rank 5 sits inside the grid, so its neighbours are the same whatever the periodicity. Rank 0
is in a corner: with periodic axes its lower neighbours are the blocks at the far ends (the
arrows leave the array and come back on the other side), without them they are walls
(`PROC_NULL`, a negative number in the tuples above: $-2$ under Open MPI, $-1$ under MPICH).
With
only two ranks along an axis the two neighbours are the same rank, and with one rank along a
periodic axis a rank is its own neighbour.

Only the neighbours along the axes are listed. The diagonal neighbours, which fill the
corner halo cells, are reached either through two exchanges, one per axis, or directly with
`update_halos(wait=False)`.

## Halo cells: what each rank stores

Each rank stores its block with `halo[axis]` extra cells on both sides of every axis, in one
NumPy or CuPy array of `layout.storage_shape`: `a.local_with_halos` is the whole storage,
`a.local` the block inside it (`layout.interior` is the slice between the two). The halo
cells are copies of the neighbours' boundary cells, filled by `update_halos()`, and
`accumulate_halos()` adds them into the neighbours' cells instead.

The figure shows a $12 \times 12$ array on $3 \times 3$ ranks with two halo layers,
periodic along axis 0 and with walls along axis 1. On the left, the halo cells of one rank
are marked in the global array; on the right, that rank's storage, with every halo cell
coloured by the rank it is filled from:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9), gridspec_kw={"width_ratios": [1.15, 1]})
for (left, right), r in zip(axes, (4, 0), strict=True):
    lay = layouts(9, (12, 12), split=(0, 1), halo=2, periodic=(True, False))[r]
    draw_blocks(left, lay, fontsize=9)
    draw_halo(left, lay)
    caption(left, f"the halo cells of rank {r} in the global array")
    draw_storage(right, lay)
    caption(
        right,
        f"the storage of rank {r}: {lay.storage_shape} for a {lay.local_shape} block",
    )
fig.tight_layout()
plt.show()

Rank 4, in the middle, is filled from all eight neighbours, the corners from the diagonal
ones. Rank 0 is in a corner: along the periodic axis 0 its upper halo rows wrap around and
come from rank 6 at the bottom of the array, while along axis 1 its left halo columns lie
beyond a wall (hatched): no neighbour fills them, so `update_halos()` leaves them as they
are, unless a `boundary=` condition is given, and `accumulate_halos()` drops what is written
into them. The [halo exchange guide](/mpiarray/guides/halo-exchange/) covers both
operations and the boundary conditions.

This is also why a halo must not be wider than the smallest block along its axis: the halo
cells are filled from the neighbouring block only, not from the one behind it.

## Uneven blocks

The near-even cut is the default, not a requirement. Three ways to cut differently; the
ranks are still placed row-major on the grid, and neighbours and halo cells work the same:

- `bounds=` gives the cut points per axis directly (`None` for an axis that is not split);
  the process grid follows from them.
- `layout.aligned(shape)` makes a layout for another shape whose blocks start where this
  one's do, for arrays that belong together but differ in length: the values at the $n$
  cells and the $n + 1$ nodes of a grid, so that cell $k$ and node $k$ are on the same rank.
- `Layout.weighted(shape, weights)` cuts each axis so that every block gets about the same
  share of a cost, for load balancing.

In [ ]:
uneven = Layout((10, 6), comm=Ranks(3), bounds=[(0, 2, 5, 10), None])
cells = Layout(16, comm=Ranks(4))
nodes = cells.aligned(17)
print("uneven bounds:", uneven.bounds)
print("cells:", cells.bounds, " nodes:", nodes.bounds)

fig, ax = plt.subplots(figsize=(3.2, 4.4))
draw_blocks(ax, uneven, fontsize=9)
caption(ax, "bounds=[(0, 2, 5, 10), None] on 3 ranks")
plt.show()

fig, ax = plt.subplots(figsize=(9, 2.2))
for r in range(4):
    ((a0, a1),) = cells.index_bounds_of(r)
    ax.add_patch(Rectangle((a0, 0), a1 - a0, 1, facecolor=rank_color(r), alpha=0.65))
    ax.add_patch(Rectangle((a0, 0), a1 - a0, 1, fill=False, edgecolor="black"))
    ((b0, b1),) = nodes.index_bounds_of(r)
    ax.scatter(
        np.arange(b0, b1),
        np.full(b1 - b0, 1.5),
        s=70,
        color=rank_color(r),
        edgecolor="black",
        zorder=3,
    )
ax.set(xlim=(-0.5, 17.5), ylim=(-0.3, 2.0), yticks=[0.5, 1.5], xticks=range(0, 17, 2))
ax.set_yticklabels(["cells: Layout(16)", "nodes: .aligned(17)"])
ax.set_xlabel("global index")
ax.grid(visible=False)
plt.show()

Cell $k$ and node $k$ are on the same rank, and the last block takes the extra node.

`Layout.weighted` sums the weights over the other axes to get one cost profile per split
axis and places the cuts where the cumulative cost crosses equal shares (a block keeps at
least one element, and at least its halo width). When the first rows of a $24 \times 24$
array cost three times as much as the rest, for example because more particles live there,
the near-even cut leaves rank 0 with nearly half the total cost, and the weighted one evens it out:

In [ ]:
n = 24
cost = np.ones((n, n))
cost[:8] = 3.0  # the first 8 rows cost three times as much per cell
even = layouts(4, (n, n))[0]
balanced = Layout.weighted((n, n), cost, comm=Ranks(4))
print("near-even rows:", even.bounds[0], " weighted rows:", balanced.bounds[0])

fig, axes = plt.subplots(
    1, 3, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1, 1, 1.1]}
)
for ax, lay, title in zip(
    axes[:2], (even, balanced), ("near-even", "Layout.weighted"), strict=True
):
    ax.imshow(cost, cmap="Greys", vmin=0, vmax=4, extent=(0, n, n, 0))
    for c in lay.bounds[0][1:-1]:
        ax.axhline(c, color=ACCENT, linewidth=2.5)
    for r in range(4):
        (a0, a1), _ = lay.index_bounds_of(r)
        ax.text(
            n / 2,
            (a0 + a1) / 2,
            f"rank {r}",
            ha="center",
            va="center",
            color="white",
            fontsize=9,
        )
    ax.set(xticks=[], yticks=lay.bounds[0], ylabel="axis 0")
    ax.grid(visible=False)
    caption(ax, f"{title}: cuts at {lay.bounds[0]}")
totals = {
    name: [cost[slice(*lay.index_bounds_of(r)[0])].sum() for r in range(4)]
    for name, lay in (("near-even", even), ("weighted", balanced))
}
x = np.arange(4)
axes[2].bar(x - 0.2, totals["near-even"], 0.4, label="near-even", color="#b0b0b0")
axes[2].bar(x + 0.2, totals["weighted"], 0.4, label="weighted", color=ACCENT)
axes[2].set(xticks=x, xlabel="rank", ylabel="total cost of the block")
axes[2].legend()
fig.tight_layout()
plt.show()

`a.rebalance(weights)` redistributes an existing array to such a layout, and
`a.redistribute(layout)` to any other layout of the same shape, in one `Alltoallv`; see
[Changing the layout](/mpiarray/guides/layouts/#changing-the-layout). Arrays combine
elementwise only when their layouts are equal, cut points included.

## The decomposition of this run

Everything above is computed by `Layout` from the number of ranks, so the real thing looks
the same. This notebook runs on several ranks when the docs are built; the cell below asks
each rank for its block and neighbours (`allgather` is collective, so every rank calls it)
and draws the layout of this run:

In [ ]:
a = mpa.zeros((12, 9), split=(0, 1), halo=1, periodic=(True, False))
layout = a.layout
rows = comm.allgather(
    (layout.rank, layout.process_coord, layout.index_bounds, layout.neighbours)
)


def name(neighbour):
    return "wall" if neighbour == PROC_NULL else str(neighbour)


if rank == 0:
    print(
        f"{nranks} rank(s), process grid {layout.process_grid}, bounds {layout.bounds}"
    )
    for r, coord, bounds, neighbours in sorted(rows):
        pairs = ", ".join(f"({name(lo)}, {name(hi)})" for lo, hi in neighbours)
        print(f"rank {r} at {coord} owns {bounds}; neighbours {pairs}")
    fig, ax = plt.subplots(figsize=(4, 4.8))
    draw_blocks(ax, layout, coords=True)
    caption(ax, f"this run: {nranks} rank(s), {a.layout.local_shape} cells on rank 0")
    plt.show()

## Explore

The explorer below applies the same rules in the browser: choose the number of ranks, the
shape, the split axes, the halo width, the periodicity and, instead of the automatic
process grid, any other factorisation. Hover a block to see that rank's process coordinates,
index bounds, storage and neighbours (click to keep it); the dashed cells are its halo
cells, and the arrows point to the neighbours that fill them. Settings that mpiarray rejects
show the `ValueError` it would raise.

In [ ]:
from importlib.util import find_spec

explorer = r"""
<div class="mpa-explorer" id="mpa-explorer">
<style>
.mpa-explorer { --mpa-text: var(--sl-color-text, #222); --mpa-line: var(--sl-color-hairline, #c8c8c8); --mpa-accent: var(--sl-color-accent, #2a78d6); --mpa-bg: var(--sl-color-bg, #fff); --mpa-soft: var(--sl-color-gray-6, #f3f3f3);
  font-family: var(--sl-font, system-ui, sans-serif); color: var(--mpa-text); border: 1px solid var(--mpa-line); border-radius: 0.75rem; padding: 0.75rem 1rem 1rem; margin-block: 1rem; }
.mpa-explorer * { box-sizing: border-box; }
.mpa-controls { display: grid; grid-template-columns: repeat(auto-fit, minmax(13rem, 1fr)); gap: 0.4rem 1.25rem; font-size: 0.9rem; margin-bottom: 0.75rem; }
.mpa-controls label { display: flex; align-items: center; gap: 0.5rem; white-space: nowrap; }
.mpa-controls label > span:first-child { min-width: 6.5rem; }
.mpa-controls input[type=range] { flex: 1; min-width: 4rem; accent-color: var(--mpa-accent); }
.mpa-controls output { min-width: 1.6rem; text-align: right; font-variant-numeric: tabular-nums; }
.mpa-controls select { font: inherit; color: inherit; background: var(--mpa-bg); border: 1px solid var(--mpa-line); border-radius: 0.3rem; padding: 0.1rem 0.3rem; }
.mpa-controls .mpa-checks { display: flex; gap: 1rem; }
.mpa-controls .mpa-checks label { gap: 0.3rem; }
.mpa-panels { display: grid; grid-template-columns: minmax(0, 3fr) minmax(0, 2fr); gap: 1rem; align-items: start; }
@media (max-width: 50rem) { .mpa-panels { grid-template-columns: 1fr; } }
.mpa-panel h4 { margin: 0 0 0.3rem; font-size: 0.85rem; font-weight: 600; color: var(--mpa-text); }
.mpa-explorer svg { width: 100%; height: auto; display: block; }
.mpa-explorer svg text { font-family: inherit; fill: var(--mpa-text); }
.mpa-explorer svg .mpa-cell { stroke: var(--mpa-bg); stroke-width: 0.04; }
.mpa-explorer svg .mpa-cut { stroke: var(--mpa-text); stroke-width: 0.12; fill: none; }
.mpa-explorer svg .mpa-block { fill: transparent; cursor: pointer; }
.mpa-explorer svg .mpa-block:hover { stroke: var(--mpa-accent); stroke-width: 0.15; }
.mpa-explorer svg .mpa-selected { fill: none; stroke: var(--mpa-accent); stroke-width: 0.22; pointer-events: none; }
.mpa-explorer svg .mpa-halo { fill: var(--mpa-accent); fill-opacity: 0.18; stroke: var(--mpa-accent); stroke-width: 0.08; stroke-dasharray: 0.25 0.15; pointer-events: none; }
.mpa-explorer svg .mpa-wall { fill: url(#mpa-hatch); stroke: var(--mpa-line); stroke-width: 0.05; pointer-events: none; }
.mpa-explorer svg .mpa-arrow { stroke: var(--mpa-text); stroke-width: 0.1; fill: none; pointer-events: none; }
.mpa-explorer svg .mpa-arrowhead { fill: var(--mpa-text); }
.mpa-explorer svg .mpa-label { font-size: 0.55px; text-anchor: middle; dominant-baseline: central; pointer-events: none; font-weight: 600; }
.mpa-explorer svg .mpa-tick { font-size: 0.42px; text-anchor: middle; dominant-baseline: central; fill: var(--mpa-text); opacity: 0.75; }
.mpa-explorer svg .mpa-tag { font-size: 0.4px; text-anchor: middle; dominant-baseline: central; font-weight: 600; pointer-events: none; }
.mpa-info { font-family: var(--sl-font-mono, ui-monospace, monospace); font-size: 0.78rem; line-height: 1.5; white-space: pre-wrap; overflow-wrap: anywhere; margin: 0; padding: 0.6rem 0.8rem; background: var(--mpa-soft); border-radius: 0.5rem; }
.mpa-error { color: #c0392b; font-weight: 600; white-space: normal; }
.mpa-hint { font-size: 0.8rem; opacity: 0.75; margin: 0.5rem 0 0; }
</style>
<div class="mpa-controls">
  <label><span>ranks</span><input type="range" id="mpa-size" min="1" max="64" value="6"><output>6</output></label>
  <label><span>axis 0 length</span><input type="range" id="mpa-n0" min="1" max="48" value="12"><output>12</output></label>
  <label><span>axis 1 length</span><input type="range" id="mpa-n1" min="1" max="48" value="9"><output>9</output></label>
  <label><span>halo</span><input type="range" id="mpa-halo" min="0" max="4" value="1"><output>1</output></label>
  <div class="mpa-checks"><span style="min-width:6.5rem">split</span>
    <label><input type="checkbox" id="mpa-split0" checked>axis 0</label>
    <label><input type="checkbox" id="mpa-split1" checked>axis 1</label></div>
  <div class="mpa-checks"><span style="min-width:6.5rem">periodic</span>
    <label><input type="checkbox" id="mpa-per0" checked>axis 0</label>
    <label><input type="checkbox" id="mpa-per1">axis 1</label></div>
  <label><span>process grid</span><select id="mpa-grid"><option value="auto">automatic</option></select></label>
</div>
<div class="mpa-panels">
  <div class="mpa-panel"><h4 id="mpa-title-array">the global array, coloured by the rank that owns each cell</h4><svg id="mpa-array" xmlns="http://www.w3.org/2000/svg"></svg></div>
  <div class="mpa-panel"><h4 id="mpa-title-storage">one rank's storage</h4><svg id="mpa-storage" xmlns="http://www.w3.org/2000/svg"></svg>
    <pre class="mpa-info" id="mpa-info"></pre></div>
</div>
<p class="mpa-hint">Hover a block to see that rank's view; click to keep it. Dashed: the halo cells of the selected rank; hatched: halo cells outside a wall, which no neighbour fills.</p>
</div>
<script>
(function () {
// The decomposition of mpiarray's Layout, in JavaScript: the same rules as
// mpiarray/layout.py (process_grid, the fitting grid, chunk_bounds, the row-major
// rank numbering and the neighbours), for the interactive explorer of the docs.
const PROC_NULL = -2;

function chunkBounds(length, nchunks, i) {
  const base = Math.floor(length / nchunks);
  const extra = length % nchunks;
  const start = i * base + Math.min(i, extra);
  return [start, start + base + (i < extra ? 1 : 0)];
}

function processGrid(size, ndim, split) {
  const grid = new Array(ndim).fill(1);
  let remaining = size;
  const axes = [...new Set(split)].sort((a, b) => a - b);
  axes.forEach((axis, position) => {
    const remainingAxes = axes.length - position;
    let target = Math.round(remaining ** (1 / remainingAxes));
    while (remaining % target !== 0) target -= 1;
    grid[axis] = target;
    remaining = Math.floor(remaining / target);
  });
  return grid;
}

// The grid that fits (at most shape[axis] ranks per split axis) with the smallest
// halo surface, or null.
function fittingGrid(size, shape, split) {
  const axes = [...new Set(split)].sort((a, b) => a - b);
  let best = null;
  const grid = new Array(shape.length).fill(1);
  const search = (position, remaining) => {
    if (position === axes.length) {
      if (remaining !== 1) return;
      const blocks = shape.map((length, axis) => Math.ceil(length / grid[axis]));
      let surface = 0;
      for (const axis of axes) {
        if (grid[axis] > 1) {
          surface += blocks.reduce((p, b, a) => (a === axis ? p : p * b), 1);
        }
      }
      const candidate = [surface, [...grid]];
      if (best === null || lessThan(candidate, best)) best = candidate;
      return;
    }
    const axis = axes[position];
    for (let n = 1; n <= Math.min(remaining, shape[axis]); n++) {
      if (remaining % n === 0) {
        grid[axis] = n;
        search(position + 1, remaining / n);
      }
    }
    grid[axis] = 1;
  };
  search(0, size);
  return best === null ? null : best[1];
}

// Python's tuple comparison of (surface, grid).
function lessThan(a, b) {
  if (a[0] !== b[0]) return a[0] < b[0];
  for (let i = 0; i < a[1].length; i++) {
    if (a[1][i] !== b[1][i]) return a[1][i] < b[1][i];
  }
  return false;
}

function coordOf(rank, grid) {
  const coord = [];
  for (let axis = grid.length - 1; axis >= 0; axis--) {
    coord.unshift(rank % grid[axis]);
    rank = Math.floor(rank / grid[axis]);
  }
  return coord;
}

function rankOf(coord, grid) {
  return coord.reduce((rank, c, axis) => rank * grid[axis] + c, 0);
}

// The decomposition of `shape` over `size` ranks, or {error}.
function decompose({ size, shape, split, halo, periodic, processGrid: given }) {
  const ndim = shape.length;
  let grid;
  const splitAxes = given ? given.map((n, a) => (n > 1 ? a : -1)).filter((a) => a >= 0) : split;
  if (given) {
    grid = [...given];
    if (grid.reduce((p, n) => p * n, 1) !== size) {
      return { error: `process_grid (${grid.join(', ')}) does not multiply to the ${size} ranks` };
    }
  } else {
    grid = processGrid(size, ndim, split);
    if (grid.some((n, axis) => n > shape[axis])) grid = fittingGrid(size, shape, split) || grid;
  }
  const cuts = shape.map((length, axis) => {
    const n = grid[axis];
    const c = [];
    for (let i = 0; i < n; i++) c.push(chunkBounds(length, n, i)[0]);
    c.push(length);
    return c;
  });
  for (let axis = 0; axis < ndim; axis++) {
    const n = grid[axis];
    if (n > 1 && n > shape[axis]) {
      return {
        error: `axis ${axis} has ${shape[axis]} elements but is split over ${n} ranks; use fewer ranks, split=None, another split axis, or a process_grid`,
      };
    }
    let smallest = Infinity;
    for (let i = 0; i < n; i++) smallest = Math.min(smallest, cuts[axis][i + 1] - cuts[axis][i]);
    if (halo[axis] && (n > 1 || periodic[axis]) && smallest < halo[axis]) {
      return {
        error: `halo ${halo[axis]} along axis ${axis} is wider than the smallest block (${smallest} elements); use a smaller halo or fewer ranks`,
      };
    }
  }
  const replicated = size > 1 && grid.every((n) => n === 1);
  const ranks = [];
  for (let rank = 0; rank < size; rank++) {
    const coord = replicated ? new Array(ndim).fill(0) : coordOf(rank, grid);
    const bounds = coord.map((c, axis) => [cuts[axis][c], cuts[axis][c + 1]]);
    const neighbours = [];
    for (let axis = 0; axis < ndim; axis++) {
      if (replicated) {
        neighbours.push(periodic[axis] ? [rank, rank] : [PROC_NULL, PROC_NULL]);
        continue;
      }
      const pair = [-1, +1].map((step) => {
        const shifted = [...coord];
        shifted[axis] += step;
        if (shifted[axis] < 0 || shifted[axis] >= grid[axis]) {
          if (!periodic[axis]) return PROC_NULL;
          shifted[axis] = ((shifted[axis] % grid[axis]) + grid[axis]) % grid[axis];
        }
        return rankOf(shifted, grid);
      });
      neighbours.push(pair);
    }
    ranks.push({ rank, coord, bounds, neighbours });
  }
  return { grid, cuts, split: splitAxes, replicated, ranks };
}

const PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5"];
const root = document.getElementById('mpa-explorer');
const $ = (id) => root.querySelector('#' + id);
const controls = ['mpa-size', 'mpa-n0', 'mpa-n1', 'mpa-halo', 'mpa-split0', 'mpa-split1', 'mpa-per0', 'mpa-per1', 'mpa-grid'];
let selected = 0;
let pinned = false;
let lastSize = 0;

const rankColor = (rank) => PALETTE[rank % PALETTE.length];
const tuple = (items) => '(' + items.join(', ') + (items.length === 1 ? ',' : '') + ')';
const pyBool = (b) => (b ? 'True' : 'False');
const neighbourName = (n) => (n === PROC_NULL ? 'PROC_NULL' : String(n));

function settings() {
  const split = [];
  if ($('mpa-split0').checked) split.push(0);
  if ($('mpa-split1').checked) split.push(1);
  const size = +$('mpa-size').value;
  const gridChoice = $('mpa-grid').value;
  return {
    size,
    shape: [+$('mpa-n0').value, +$('mpa-n1').value],
    split,
    halo: [+$('mpa-halo').value, +$('mpa-halo').value],
    periodic: [$('mpa-per0').checked, $('mpa-per1').checked],
    processGrid: gridChoice === 'auto' ? null : gridChoice.split('x').map(Number),
  };
}

// The possible process grids for the rank count: every factorisation into two axes.
function refreshGridChoices(size) {
  if (size === lastSize) return;
  lastSize = size;
  const select = $('mpa-grid');
  const current = select.value;
  select.replaceChildren();
  const auto = document.createElement('option');
  auto.value = 'auto';
  auto.textContent = 'automatic';
  select.appendChild(auto);
  for (let a = 1; a <= size; a++) {
    if (size % a) continue;
    const option = document.createElement('option');
    option.value = a + 'x' + size / a;
    option.textContent = a + ' × ' + size / a;
    select.appendChild(option);
  }
  select.value = [...select.options].some((o) => o.value === current) ? current : 'auto';
}

function el(name, attributes, text) {
  const node = document.createElementNS('http://www.w3.org/2000/svg', name);
  for (const [key, value] of Object.entries(attributes)) node.setAttribute(key, value);
  if (text !== undefined) node.textContent = text;
  return node;
}

function hatch(svg) {
  const defs = el('defs', {});
  const pattern = el('pattern', { id: 'mpa-hatch', width: '0.5', height: '0.5', patternUnits: 'userSpaceOnUse', patternTransform: 'rotate(45)' });
  pattern.appendChild(el('rect', { width: '0.5', height: '0.5', fill: 'var(--mpa-soft)' }));
  pattern.appendChild(el('line', { x1: 0, y1: 0, x2: 0, y2: 0.5, stroke: 'var(--mpa-line)', 'stroke-width': 0.12 }));
  defs.appendChild(pattern);
  svg.appendChild(defs);
}

// The boxes of halo cells of a rank, in global indices, each with its source: a rank
// (after wrapping along periodic axes) or 'wall'.
function haloBoxes(result, rankInfo, halo, periodic, shape) {
  const boxes = [];
  for (const d0 of [-1, 0, 1]) {
    for (const d1 of [-1, 0, 1]) {
      if (d0 === 0 && d1 === 0) continue;
      const box = [];
      let wall = false;
      let wrapped = [];
      [d0, d1].forEach((d, axis) => {
        const [start, end] = rankInfo.bounds[axis];
        const h = halo[axis];
        let range = d === 0 ? [start, end] : d < 0 ? [start - h, start] : [end, end + h];
        if (range[0] === range[1]) return;
        let shift = 0;
        if (range[0] < 0 || range[1] > shape[axis]) {
          if (periodic[axis]) shift = range[0] < 0 ? shape[axis] : -shape[axis];
          else wall = true;
        }
        wrapped.push(shift);
        box.push(range);
      });
      if (box.length < 2) continue;
      const inside = box.map((r, axis) => [r[0] + wrapped[axis], r[1] + wrapped[axis]]);
      let source = 'wall';
      if (!wall) {
        // the owner of the first cell of the (wrapped) box
        const cell = inside.map((r) => r[0]);
        source = result.ranks.find((r) => cell.every((c, a) => c >= r.bounds[a][0] && c < r.bounds[a][1])).rank;
      }
      boxes.push({ box, inside, source, wall, offset: [d0, d1] });
    }
  }
  return boxes;
}

function drawArray(result, s) {
  const svg = $('mpa-array');
  svg.replaceChildren();
  const [n0, n1] = s.shape;
  const h = Math.max(...s.halo);
  const margin = h + 1.5;
  svg.setAttribute('viewBox', `${-margin} ${-margin} ${n1 + 2 * margin} ${n0 + 2 * margin}`);
  hatch(svg);
  if (result.error) {
    svg.appendChild(el('rect', { x: 0, y: 0, width: n1, height: n0, class: 'mpa-wall' }));
    return;
  }
  const selectedInfo = result.ranks[Math.min(selected, s.size - 1)];
  // cells
  for (const r of result.ranks) {
    const [[a0, a1], [b0, b1]] = r.bounds;
    const fill = result.replicated ? 'var(--mpa-soft)' : rankColor(r.rank);
    for (let i = a0; i < a1; i++) {
      for (let j = b0; j < b1; j++) {
        svg.appendChild(el('rect', { x: j, y: i, width: 1, height: 1, fill, 'fill-opacity': 0.65, class: 'mpa-cell' }));
      }
    }
  }
  // the halo of the selected rank
  const boxes = haloBoxes(result, selectedInfo, s.halo, s.periodic, s.shape);
  for (const { box, inside, wall } of boxes) {
    const target = wall ? box : inside;
    const [[a0, a1], [b0, b1]] = target;
    svg.appendChild(el('rect', { x: b0, y: a0, width: b1 - b0, height: a1 - a0, class: wall ? 'mpa-wall' : 'mpa-halo' }));
  }
  // cuts and the outline
  for (const c of result.cuts[0].slice(1, -1)) svg.appendChild(el('line', { x1: 0, y1: c, x2: n1, y2: c, class: 'mpa-cut' }));
  for (const c of result.cuts[1].slice(1, -1)) svg.appendChild(el('line', { x1: c, y1: 0, x2: c, y2: n0, class: 'mpa-cut' }));
  svg.appendChild(el('rect', { x: 0, y: 0, width: n1, height: n0, class: 'mpa-cut' }));
  // ticks: where the blocks start along each axis, and the length
  for (const c of result.cuts[1]) svg.appendChild(el('text', { x: c, y: -0.45, class: 'mpa-tick' }, c));
  for (const c of result.cuts[0]) svg.appendChild(el('text', { x: -0.45, y: c, class: 'mpa-tick' }, c));
  // labels, neighbour tags and hit areas
  const fontSize = Math.max(0.3, Math.min(0.7, 0.35 * Math.min(...result.ranks.map((r) => Math.min(r.bounds[0][1] - r.bounds[0][0], r.bounds[1][1] - r.bounds[1][0])))));
  for (const r of result.ranks) {
    const [[a0, a1], [b0, b1]] = r.bounds;
    const label = result.replicated ? 'every rank' : String(r.rank);
    svg.appendChild(el('text', { x: (b0 + b1) / 2, y: (a0 + a1) / 2, class: 'mpa-label', style: `font-size:${result.replicated ? 0.5 : fontSize}px` }, label));
  }
  if (!result.replicated) {
    const [[a0, a1], [b0, b1]] = selectedInfo.bounds;
    svg.appendChild(el('rect', { x: b0, y: a0, width: b1 - b0, height: a1 - a0, class: 'mpa-selected' }));
    const centre = [(b0 + b1) / 2, (a0 + a1) / 2];
    const tags = {};
    selectedInfo.neighbours.forEach((pair, axis) => {
      pair.forEach((neighbour, side) => {
        const step = side === 0 ? -1 : 1;
        if (neighbour === PROC_NULL) return;
        const target = result.ranks[neighbour];
        const [[c0, c1], [d0, d1]] = target.bounds;
        const tc = [(d0 + d1) / 2, (c0 + c1) / 2];
        // the neighbour's block along the axis: adjacent, or across the array (periodic)
        const adjacent = axis === 0 ? (step < 0 ? c1 === a0 : c0 === a1) : step < 0 ? d1 === b0 : d0 === b1;
        let path;
        const from = [...centre];
        const to = [...tc];
        if (axis === 0) { from[1] = step < 0 ? a0 : a1; to[1] = step < 0 ? c1 : c0; } else { from[0] = step < 0 ? b0 : b1; to[0] = step < 0 ? d1 : d0; }
        if (adjacent || neighbour === selectedInfo.rank) {
          if (neighbour === selectedInfo.rank) {
            // one rank along a periodic axis: its own neighbour; a loop out of the array and back
            const out = axis === 0 ? [from[0], step < 0 ? -h - 0.6 : n0 + h + 0.6] : [step < 0 ? -h - 0.6 : n1 + h + 0.6, from[1]];
            path = `M ${from[0]} ${from[1]} L ${out[0]} ${out[1]}`;
          } else {
            path = `M ${from[0]} ${from[1]} L ${to[0]} ${to[1]}`;
          }
        } else {
          // wrap-around: out of the array on this side, in again on the other
          const outA = axis === 0 ? [from[0], step < 0 ? -h - 0.6 : n0 + h + 0.6] : [step < 0 ? -h - 0.6 : n1 + h + 0.6, from[1]];
          const outB = axis === 0 ? [to[0], step < 0 ? n0 + h + 0.6 : -h - 0.6] : [step < 0 ? n1 + h + 0.6 : -h - 0.6, to[1]];
          path = `M ${from[0]} ${from[1]} L ${outA[0]} ${outA[1]} M ${outB[0]} ${outB[1]} L ${to[0]} ${to[1]}`;
        }
        svg.appendChild(el('path', { d: path, class: 'mpa-arrow', 'marker-end': 'url(#mpa-head)' }));
        if (neighbour !== selectedInfo.rank) {
          (tags[neighbour] ??= []).push(`axis ${axis} ${step < 0 ? '−' : '+'}`);
        }
      });
    });
    for (const [neighbour, names] of Object.entries(tags)) {
      const [[c0, c1], [d0, d1]] = result.ranks[neighbour].bounds;
      svg.appendChild(el('text', { x: (d0 + d1) / 2, y: (c0 + c1) / 2 + fontSize * 0.9, class: 'mpa-tag' }, names.join(', ')));
    }
    const marker = el('marker', { id: 'mpa-head', viewBox: '0 0 10 10', refX: 8, refY: 5, markerWidth: 4, markerHeight: 4, orient: 'auto-start-reverse' });
    marker.appendChild(el('path', { d: 'M 0 0 L 10 5 L 0 10 z', class: 'mpa-arrowhead' }));
    svg.querySelector('defs').appendChild(marker);
    // walls: no neighbour, so the halo cells there are never filled
    selectedInfo.neighbours.forEach((pair, axis) => {
      pair.forEach((neighbour, side) => {
        if (neighbour !== PROC_NULL) return;
        const x = axis === 0 ? (b0 + b1) / 2 : side === 0 ? -h - 0.6 : n1 + h + 0.6;
        const y = axis === 0 ? (side === 0 ? -h - 0.6 : n0 + h + 0.6) : (a0 + a1) / 2;
        const rotate = axis === 0 ? '' : `rotate(${side === 0 ? -90 : 90} ${x} ${y})`;
        svg.appendChild(el('text', { x, y, class: 'mpa-tag', style: 'opacity:0.7', transform: rotate }, 'PROC_NULL'));
      });
    });
  }
  for (const r of result.ranks) {
    const [[a0, a1], [b0, b1]] = r.bounds;
    const hit = el('rect', { x: b0, y: a0, width: b1 - b0, height: a1 - a0, class: 'mpa-block' });
    hit.addEventListener('pointerenter', () => { if (!pinned) { selected = r.rank; render(); } });
    hit.addEventListener('click', () => { pinned = !(pinned && selected === r.rank); selected = r.rank; render(); });
    svg.appendChild(hit);
  }
}

function drawStorage(result, s) {
  const svg = $('mpa-storage');
  svg.replaceChildren();
  if (result.error) { svg.setAttribute('viewBox', '0 0 1 1'); return; }
  const info = result.ranks[Math.min(selected, s.size - 1)];
  const [h0, h1] = s.halo;
  const [[a0, a1], [b0, b1]] = info.bounds;
  const m0 = a1 - a0 + 2 * h0, m1 = b1 - b0 + 2 * h1;
  svg.setAttribute('viewBox', `-0.9 -0.9 ${m1 + 1.8} ${m0 + 1.8}`);
  hatch(svg);
  const boxes = haloBoxes(result, info, s.halo, s.periodic, s.shape);
  for (let i = 0; i < m0; i++) {
    for (let j = 0; j < m1; j++) {
      const g = [a0 - h0 + i, b0 - h1 + j];
      const own = g[0] >= a0 && g[0] < a1 && g[1] >= b0 && g[1] < b1;
      let fill = own ? (result.replicated ? 'var(--mpa-soft)' : rankColor(info.rank)) : null;
      let wall = false;
      if (!own) {
        const box = boxes.find(({ box: bx }) => g[0] >= bx[0][0] && g[0] < bx[0][1] && g[1] >= bx[1][0] && g[1] < bx[1][1]);
        if (box && !box.wall) fill = result.replicated ? 'var(--mpa-soft)' : rankColor(box.source);
        else wall = true;
      }
      svg.appendChild(el('rect', { x: j, y: i, width: 1, height: 1, fill: wall ? 'url(#mpa-hatch)' : fill, 'fill-opacity': wall ? 1 : own ? 0.65 : 0.45, class: 'mpa-cell' }));
    }
  }
  svg.appendChild(el('rect', { x: h1, y: h0, width: b1 - b0, height: a1 - a0, class: 'mpa-cut' }));
  svg.appendChild(el('rect', { x: 0, y: 0, width: m1, height: m0, class: 'mpa-cut', style: 'stroke-dasharray:0.25 0.15; stroke-width:0.06' }));
  // storage indices along the edges
  for (const j of [0, h1, m1 - h1, m1]) if (j <= m1) svg.appendChild(el('text', { x: j, y: -0.45, class: 'mpa-tick' }, j));
  for (const i of [0, h0, m0 - h0, m0]) if (i <= m0) svg.appendChild(el('text', { x: -0.45, y: i, class: 'mpa-tick' }, i));
  const fontSize = Math.max(0.3, Math.min(0.7, 0.35 * Math.min(a1 - a0, b1 - b0)));
  svg.appendChild(el('text', { x: h1 + (b1 - b0) / 2, y: h0 + (a1 - a0) / 2, class: 'mpa-label', style: `font-size:${fontSize}px` }, result.replicated ? 'every rank' : `local (rank ${info.rank})`));
}

function describe(result, s) {
  const info = $('mpa-info');
  const splitRepr = s.split.length === 0 ? 'None' : s.split.length === 1 ? String(s.split[0]) : tuple(s.split);
  const haloRepr = s.halo[0];
  const periodicRepr = s.periodic[0] === s.periodic[1] ? pyBool(s.periodic[0]) : tuple(s.periodic.map(pyBool));
  const gridArg = s.processGrid ? `, process_grid=${tuple(s.processGrid)}` : '';
  const call = `mpa.zeros(${tuple(s.shape)}, split=${splitRepr}, halo=${haloRepr}, periodic=${periodicRepr}${gridArg})`;
  if (result.error) {
    info.innerHTML = `# mpiexec -n ${s.size} python script.py\n${call}\n\n<span class="mpa-error">ValueError: ${result.error}</span>`;
    $('mpa-title-storage').textContent = 'no layout';
    return;
  }
  const r = result.ranks[Math.min(selected, s.size - 1)];
  const local = r.bounds.map(([a, b]) => b - a);
  const storage = local.map((n, axis) => n + 2 * s.halo[axis]);
  const lines = [
    `# mpiexec -n ${s.size} python script.py`,
    `a = ${call}`,
    `a.layout.process_grid   ${tuple(result.grid)}`,
    `a.layout.bounds         ${tuple(result.cuts.map((c) => tuple(c)))}`,
    `a.layout.replicated     ${pyBool(result.replicated)}`,
    '',
    `# on rank ${r.rank}`,
    `a.layout.process_coord  ${tuple(r.coord)}`,
    `a.layout.index_bounds   ${tuple(r.bounds.map((b) => tuple(b)))}`,
    `a.layout.local_shape    ${tuple(local)}`,
    `a.layout.storage_shape  ${tuple(storage)}`,
    `a.layout.neighbours     ${tuple(r.neighbours.map((pair) => tuple(pair.map(neighbourName))))}`,
  ];
  info.textContent = lines.join('\n');
  $('mpa-title-storage').textContent = result.replicated ? 'the storage of every rank (local_with_halos)' : `the storage of rank ${r.rank} (local_with_halos)`;
}

function render() {
  const s = settings();
  refreshGridChoices(s.size);
  const current = settings();
  for (const id of ['mpa-size', 'mpa-n0', 'mpa-n1', 'mpa-halo']) $(id).nextElementSibling.value = $(id).value;
  const result = decompose(current);
  if (!result.error && selected >= current.size) selected = 0;
  drawArray(result, current);
  drawStorage(result, current);
  describe(result, current);
}

for (const id of controls) $(id).addEventListener('input', render);
render();
})();
</script>

"""
if find_spec(
    "IPython"
):  # a notebook or the docs page; a plain script has nowhere to show it
    from IPython.display import HTML, display

    display(HTML(explorer))

## Summary

| Step | Decided by | Read from the layout |
| ---- | ---------- | -------------------- |
| ranks per axis | `split` and the number of ranks (`mpa.process_grid`), or `process_grid=`; a fitting grid if that one does not fit | `process_grid` |
| cut points per axis | `mpa.chunk_bounds`, or `bounds=`, `aligned`, `Layout.weighted` | `bounds`, `index_bounds`, `owner`, `owners` |
| position of each rank | row-major over the grid, or MPI with `reorder=True` | `rank`, `process_coord`, `coord_of`, `rank_of` |
| neighbours | adjacent blocks; `periodic` at the ends, else `PROC_NULL` | `neighbours` |
| storage | the block plus `halo` cells on both sides of every axis | `local_shape`, `storage_shape`, `interior` |

Next: [Halo exchange](/mpiarray/guides/halo-exchange/) moves data through the halo cells, and
the [heat equation](/mpiarray/guides/heat-equation/) puts the decomposition to work.